<a href="https://colab.research.google.com/github/nisabulut/digital-image-processing/blob/main/5_4_goruntu_isleme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!sudo apt-get update
!sudo apt-get install libopencv-dev

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopencv-dev is already the newest version (4.6.0+dfsg-13.1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [5]:
%%writefile color_space.cpp
#include <opencv2/opencv.hpp>
#include <iostream>
#include <cmath>

// 1. Fonksiyon Pointer'ı Tanımlaması (Typedef)
// Girdi olarak 3 kanallı (BGR) piksel alır, çıktı olarak dönüştürülmüş pikseli döndürür.
typedef cv::Vec3b (*PixelTransformFunc)(const cv::Vec3b&);

// 2. Dönüşüm Fonksiyonu (BGR -> RGB)
// OpenCV görselleri BGR okur. Biz bunu RGB'ye çevirmek için 0. ve 2. indeksleri yer değiştiriyoruz.
cv::Vec3b convertBgrToRgb(const cv::Vec3b& pixel) {
    return cv::Vec3b(pixel[2], pixel[1], pixel[0]);
}

// 3. Fonksiyon Pointer'ını Parametre Olarak Alan Ana Dönüşüm Fonksiyonu
cv::Mat applyColorSpaceTransform(const cv::Mat& input, PixelTransformFunc transformFunc) {
    cv::Mat output(input.rows, input.cols, CV_8UC3);

    for (int r = 0; r < input.rows; ++r) {
        for (int c = 0; c < input.cols; ++c) {
            // Her piksel için, parametre olarak gelen pointer (transformFunc) üzerinden dönüşümü uygula
            output.at<cv::Vec3b>(r, c) = transformFunc(input.at<cv::Vec3b>(r, c));
        }
    }
    return output;
}

int main() {
    std::string imagePath = "image.jpg"; // Görselinin adını gir
    cv::Mat image = cv::imread(imagePath, cv::IMREAD_COLOR);

    if (image.empty()) {
        std::cerr << "Hata: Gorsel yuklenemedi!" << std::endl;
        return -1;
    }

    // 4. OpenCV'nin Hazır Fonksiyonu ile Dönüşüm (Referans)
    cv::Mat rgbCV;
    cv::cvtColor(image, rgbCV, cv::COLOR_BGR2RGB);

    // 5. Kendi Fonksiyon Pointer Yapımızla Dönüşüm
    cv::Mat rgbCustom = applyColorSpaceTransform(image, convertBgrToRgb);

    // 6. Piksel Piksel Hata (Mutlak Fark) Hesaplama
    long long totalError = 0;

    for (int r = 0; r < image.rows; ++r) {
        for (int c = 0; c < image.cols; ++c) {
            cv::Vec3b pCV = rgbCV.at<cv::Vec3b>(r, c);
            cv::Vec3b pCustom = rgbCustom.at<cv::Vec3b>(r, c);

            totalError += std::abs(pCV[0] - pCustom[0]); // R kanalı farkı
            totalError += std::abs(pCV[1] - pCustom[1]); // G kanalı farkı
            totalError += std::abs(pCV[2] - pCustom[2]); // B kanalı farkı
        }
    }

    std::cout << "Gorsel Boyutu: " << image.cols << "x" << image.rows << std::endl;
    std::cout << "Toplam Hata Degeri (BGR -> RGB): " << totalError << std::endl;

    if (totalError == 0) {
        std::cout << "Mukemmel! Iki sonuc birebir ayni. Beklenen 0 hata degerine ulasildi." << std::endl;
    }

    // Çıktıları kaydetme
    cv::imwrite("rgb_cv.png", rgbCV);
    cv::imwrite("rgb_custom.png", rgbCustom);

    return 0;
}

Overwriting color_space.cpp


In [6]:
!g++ color_space.cpp -o color_space `pkg-config --cflags --libs opencv4`
!./color_space

Gorsel Boyutu: 654x520
Toplam Hata Degeri (BGR -> RGB): 0
Mukemmel! Iki sonuc birebir ayni. Beklenen 0 hata degerine ulasildi.
